## 1. Setup and Imports

In [ ]:
# Cell 1 - Installing Libraries and Importing All Packages
# This cell installs and imports all dependencies needed for ALL AI features.
%pip install rembg[gpu] gradio tensorflow tensorflow_hub opencv-python --quiet

import gradio as gr
from PIL import Image
import numpy as np
from rembg import remove, new_session
import tensorflow as tf
import tensorflow_hub as hub
import cv2
import math
import os

print("All required libraries are ready.")

## 2. Load All AI Models from Local Files

In [ ]:
# Cell 2 - Load All AI Models from the 'saved-models' Directory

# --- 1. Setup for Background Removal Models (ONNX) ---
# Point the environment variable to our local models folder for rembg
os.environ["U2NET_HOME"] = "./saved-models"
print("Preparing Background Removal model sessions from './saved-models'...")
# Pre-load the session for the anime model
anime_session = new_session("isnet-anime")
print("Background Removal models are ready.")

# --- 2. Load Super-Resolution Model (TensorFlow) ---
SR_MODEL_PATH = "./saved-models/esrgan-tf2"
print(f"Loading Super-Resolution model from local path: {SR_MODEL_PATH}...")
if os.path.exists(SR_MODEL_PATH):
    super_res_model = hub.load(SR_MODEL_PATH)
    print("Super-Resolution model loaded successfully.")
else:
    # Add a friendly error message if the model hasn't been saved yet
    raise FileNotFoundError(f"ESRGAN model not found at {SR_MODEL_PATH}. Please run the '02_AI_Feature_Super_Resolution.ipynb' notebook's save cell first.")

print("\nAll AI models are loaded and ready to use.")

## 3. Define All AI Feature Functions

In [ ]:
# Cell 3 - Define All AI Feature Functions

# --- Feature 1: Background Removal ---
def remove_background_feature(input_image: Image.Image, model_choice: str) -> Image.Image:
    if input_image is None: return None
    if model_choice == "Anime (isnet-anime)":
        return remove(input_image, session=anime_session)
    else:
        return remove(input_image)

# --- Feature 2: Super-Resolution ---
def upscale_and_downscale(input_image: Image.Image, target_scale: int, tile_size: int = 256, overlap: int = 32) -> Image.Image:
    if input_image is None: return None
    MODEL_BASE_SCALE = 4
    num_passes = 2 if target_scale > MODEL_BASE_SCALE else 1
    current_image = input_image
    for i in range(num_passes):
        print(f"Starting upscale pass {i+1}/{num_passes}...")
        width, height = current_image.size
        pass_output_image = Image.new('RGB', (width * MODEL_BASE_SCALE, height * MODEL_BASE_SCALE))
        step = tile_size - overlap
        for y in range(0, height, step):
            for x in range(0, width, step):
                bbox = (x, y, min(x + tile_size, width), min(y + tile_size, height))
                tile = current_image.crop(bbox)
                img_np = cv2.cvtColor(np.array(tile), cv2.COLOR_RGB2BGR)
                img_tf = tf.expand_dims(tf.cast(img_np, tf.float32), 0)
                upscaled_tensor = super_res_model(img_tf)
                upscaled_tensor = tf.clip_by_value(upscaled_tensor, 0, 255)
                upscaled_image_np = tf.cast(tf.squeeze(upscaled_tensor), tf.uint8).numpy()
                upscaled_tile = Image.fromarray(cv2.cvtColor(upscaled_image_np, cv2.COLOR_BGR2RGB))
                pass_output_image.paste(upscaled_tile, (x * MODEL_BASE_SCALE, y * MODEL_BASE_SCALE))
        current_image = pass_output_image
    print(f"Upscaling complete. Intermediate size: {current_image.width}x{current_image.height}.")
    final_width = int(input_image.width * target_scale)
    final_height = int(input_image.height * target_scale)
    if current_image.width != final_width or current_image.height != final_height:
        print(f"Downscaling to final size: {final_width}x{final_height}.")
        return current_image.resize((final_width, final_height), Image.Resampling.LANCZOS)
    else:
        return current_image

## 4. Unified Gradio User Interface

In [ ]:
# Cell 4 - Unified Gradio User Interface for All Features

# --- Master function to route to the correct AI tool ---
def process_image(input_image: Image.Image, tool_choice: str, bg_model: str, sr_scale: int, sr_model: str):
    if input_image is None:
        return None
    if tool_choice == "Background Removal":
        return remove_background_feature(input_image, bg_model)
    elif tool_choice == "Super-Resolution":
        # We ignore sr_model for now as we only have one
        return upscale_and_downscale(input_image, sr_scale)
    else:
        return input_image

# --- Function to control UI visibility ---
def update_tool_options(tool_choice):
    # This function shows/hides the correct "Advanced Options" based on the selected tool
    is_bg_remover = tool_choice == "Background Removal"
    is_super_res = tool_choice == "Super-Resolution"
    return gr.update(visible=is_bg_remover), gr.update(visible=is_super_res)

# --- The Gradio UI ---
with gr.Blocks(theme=gr.themes.Soft()) as iface:
    gr.Markdown("# 🚀 AI Features Showcase")
    gr.Markdown("A unified interface to test all developed AI features. Models are loaded from the local `saved-models` directory.")
    
    with gr.Row():
        with gr.Column(scale=2):
            input_img = gr.Image(type="pil", label="Upload Your Image")
            tool_choice = gr.Dropdown(
                ["Background Removal", "Super-Resolution"], 
                label="Choose AI Tool"
            )

            with gr.Column() as advanced_options:
                # Options for Background Removal
                bg_options = gr.Column(visible=False)
                with bg_options:
                    bg_model_dd = gr.Dropdown(["General (u2net)", "Anime (isnet-anime)"], label="Model", value="General (u2net)")
                
                # Options for Super-Resolution
                sr_options = gr.Column(visible=False)
                with sr_options:
                    sr_scale_slider = gr.Slider(minimum=2, maximum=8, step=1, label="Upscale Factor", value=4)
                    sr_model_dd = gr.Dropdown(["General (ESRGAN)", "Anime (Placeholder)"], label="Upscaler Model", value="General (ESRGAN)")

            submit_btn = gr.Button("Process Image", variant="primary")
            
        with gr.Column(scale=3):
            output_img = gr.Image(type="pil", label="Processed Image")

    # Event listener to make the UI dynamic
    tool_choice.change(
        fn=update_tool_options,
        inputs=tool_choice,
        outputs=[bg_options, sr_options]
    )

    submit_btn.click(
        fn=process_image,
        inputs=[input_img, tool_choice, bg_model_dd, sr_scale_slider, sr_model_dd],
        outputs=output_img
    )

# Launch the interface
iface.launch(share=True)